In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
import polars as pl

file_path = '/content/drive/MyDrive/Colab Notebooks/PMCSN/TravisTorrent.csv'

# Creiamo il piano di esecuzione con i parametri corretti
df_lazy = pl.scan_csv(
    file_path,
    null_values=["NA"],         # Dice a Polars di trattare le stringhe "NA" come dati mancanti (Null)
    infer_schema_length=10000   # Legge le prime 10.000 righe per capire meglio i tipi di dato
)

# Eseguiamo il collect() solo per le prime 5 righe per verificare che funzioni
print(df_lazy.head(5).collect())

shape: (5, 66)
┌───────────┬───────────┬──────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ tr_build_ ┆ gh_projec ┆ gh_is_pr ┆ gh_pr_cre ┆ … ┆ tr_origin ┆ tr_durati ┆ tr_status ┆ tr_jobs   │
│ id        ┆ t_name    ┆ ---      ┆ ated_at   ┆   ┆ al_commit ┆ on        ┆ ---       ┆ ---       │
│ ---       ┆ ---       ┆ bool     ┆ ---       ┆   ┆ ---       ┆ ---       ┆ str       ┆ str       │
│ i64       ┆ str       ┆          ┆ str       ┆   ┆ str       ┆ i64       ┆           ┆           │
╞═══════════╪═══════════╪══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 3154      ┆ rspec/rsp ┆ false    ┆ null      ┆ … ┆ 029e6972f ┆ 956       ┆ passed    ┆ [3161,    │
│           ┆ ec-core   ┆          ┆           ┆   ┆ cf719542d ┆           ┆           ┆ 3163,     │
│           ┆           ┆          ┆           ┆   ┆ eff1b2619 ┆           ┆           ┆ 3160,     │
│           ┆           ┆          ┆           ┆   ┆ d29…      ┆           ┆

In [ ]:
import polars as pl

file_path = '/content/drive/MyDrive/Colab Notebooks/PMCSN/TravisTorrent.csv'
print("Ricerca del progetto ideale: Filtro strutturale in corso...\n")

# Colonne su cui vogliamo garantire la completezza
colonne_target = [
    'tr_build_id', 'tr_job_id', 'gh_build_started_at',
    'tr_log_setup_time', 'tr_log_buildduration'
]

# 1. Completezza e Distribuzione degli Stati
df_stats = (
    df_lazy
    .group_by("gh_project_name")
    .agg([
        pl.len().alias("totale_job"),
        # Calcolo % stati (Failed, Errored, Canceled)
        ((pl.col("tr_status").eq("failed").sum() / pl.len()) * 100).round(2).alias("%_failed"),
        ((pl.col("tr_status").eq("errored").sum() / pl.len()) * 100).round(2).alias("%_errored"),
        ((pl.col("tr_status").eq("canceled").sum() / pl.len()) * 100).round(2).alias("%_canceled"),
        # Calcolo % record non nulli (media per le colonne critiche)
        (((sum((pl.col(col).is_not_null()).sum() for col in colonne_target)) / (pl.len() * len(colonne_target))) * 100).round(2).alias("score_completezza")
    ])
)

# 2. Variabilità dei Job per Build
df_builds = df_lazy.group_by(["gh_project_name", "tr_build_id"]).agg(pl.len().alias("num_jobs"))
df_variabilita = (
    df_builds
    .group_by("gh_project_name")
    .agg([
        pl.len().alias("totale_builds"),
        pl.col("num_jobs").max().alias("max_jobs_per_build"),
        pl.col("num_jobs").min().alias("min_jobs_per_build"),
        pl.col("num_jobs").std().round(2).alias("std_jobs")
    ])
)

# 3. Unione e Filtraggio Finale
candidati_ideali = (
    df_stats.join(df_variabilita, on="gh_project_name")
    # Filtro 1: Dati robusti (almeno 1000 job, altissima completezza)
    .filter(pl.col("totale_job") >= 1000)
    .filter(pl.col("score_completezza") >= 90.0)
    # Filtro 2: Variabilità arrivi (almeno 5 job in una singola build)
    .filter(pl.col("max_jobs_per_build") >= 5)
    # Filtro 3: Presenza non nulla di stati anomali (tutti maggiori di 0)
    .filter((pl.col("%_failed") > 0) & (pl.col("%_errored") > 0) & (pl.col("%_canceled") > 0))
    # Ordiniamo per completezza e variabilità
    .sort(["score_completezza", "std_jobs"], descending=[True, True])
).collect()

# Estraiamo i nomi dei top 3 progetti
top_3_projects = candidati_ideali["gh_project_name"].head(3).to_list()

with pl.Config(tbl_rows=10):
    print("--- TOP CANDIDATI STRUTTURALI ---")
    print(candidati_ideali.head(5))

Ricerca del progetto ideale: Filtro strutturale in corso...

--- TOP CANDIDATI STRUTTURALI ---
shape: (5, 10)
┌────────────┬───────────┬──────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ gh_project ┆ totale_jo ┆ %_failed ┆ %_errored ┆ … ┆ totale_bu ┆ max_jobs_ ┆ min_jobs_ ┆ std_jobs │
│ _name      ┆ b         ┆ ---      ┆ ---       ┆   ┆ ilds      ┆ per_build ┆ per_build ┆ ---      │
│ ---        ┆ ---       ┆ f64      ┆ f64       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ f64      │
│ str        ┆ u32       ┆          ┆           ┆   ┆ u32       ┆ u32       ┆ u32       ┆          │
╞════════════╪═══════════╪══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ checkstyle ┆ 46889     ┆ 7.1      ┆ 0.77      ┆ … ┆ 2899      ┆ 35        ┆ 1         ┆ 10.14    │
│ /checkstyl ┆           ┆          ┆           ┆   ┆           ┆           ┆           ┆          │
│ e          ┆           ┆          ┆           ┆   ┆           ┆           ┆     

In [ ]:
import numpy as np
import scipy.stats as stats
import pandas as pd
import polars as pl

print("\n--- TEST STATISTICO DELLE DISTRIBUZIONI SUI TOP 3 PROGETTI ---")

# Ricarichiamo in Pandas solo i dati dei 3 vincitori forzando i tipi corretti
df_test = (
    pl.scan_csv(file_path, null_values=["NA", ""], infer_schema_length=10000,
        schema_overrides={"tr_log_buildduration": pl.String, "tr_log_setup_time": pl.String})
    .filter(pl.col("gh_project_name").is_in(top_3_projects))
    # --- MODIFICA 1: Aggiungiamo 'tr_status' alla selezione per poter filtrare ---
    .select(["gh_project_name", "tr_build_id", "tr_job_id", "gh_build_started_at", "tr_log_buildduration", "tr_status"])
    .with_columns([
        pl.col("tr_log_buildduration").cast(pl.Float64, strict=False)
    ])
    .drop_nulls()
    .collect().to_pandas()
)

for proj in top_3_projects:
    print(f"\nAnalisi per: {proj}")
    df_proj = df_test[df_test["gh_project_name"] == proj].copy()

    # 1. Test Distribuzione Tempi di Build (Lognormale SOLO sui job ideali)
    # --- MODIFICA 2: Filtriamo i dati isolando solo i job con stato 'passed' ---
    df_passed = df_proj[df_proj["tr_status"] == "passed"]

    build_times = df_passed["tr_log_buildduration"].values
    build_times = build_times[build_times > 0] # Continua a funzionare, ma su un sottoinsieme

    # Fittiamo la lognormale (Rappresenta la durata IDEALE della compilazione)
    shape, loc, scale = stats.lognorm.fit(build_times, floc=0)

    # KS-Test (calcoliamo la statistica D)
    D_build, p_build = stats.kstest(build_times, 'lognorm', args=(shape, loc, scale))
    print(f"  [Build - Solo Passed] Lognormale -> Errore (D): {D_build:.4f} | p-value: {p_build:.4e}")

    # 2. Preparazione Inter-arrivi (TUTTI i job, calcolano il carico reale)
    # Convertiamo le date in modo robusto gestendo vari formati
    df_proj['timestamp'] = pd.to_datetime(df_proj['gh_build_started_at'], format='mixed', utc=True).astype('int64') // 10**9

    # Raggruppiamo per build per avere arrivi puliti
    arrivi = df_proj.groupby('tr_build_id')['timestamp'].min().sort_values().values
    inter_arrivi = np.diff(arrivi)
    inter_arrivi = inter_arrivi[inter_arrivi > 0] # Togliamo arrivi simultanei

    # Fittiamo Esponenziale
    loc_exp, scale_exp = stats.expon.fit(inter_arrivi, floc=0)
    D_arr, p_arr = stats.kstest(inter_arrivi, 'expon', args=(loc_exp, scale_exp))
    print(f"  [Arrivi - Tutti] Esponenziale -> Errore (D): {D_arr:.4f} | p-value: {p_arr:.4e}")

In [ ]:
import polars as pl
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

file_path = '/content/drive/MyDrive/Colab Notebooks/PMCSN/TravisTorrent.csv'
PROJECT_NAME = "SonarSource/sonar-java"

print(f"Estrazione dati e generazione grafici per {PROJECT_NAME}...")

# ==========================================
# 1. ESTRAZIONE DATI (Polars)
# ==========================================
df = (
    pl.scan_csv(file_path, null_values=["NA", ""], infer_schema_length=10000,
        schema_overrides={"tr_log_buildduration": pl.String, "tr_log_setup_time": pl.String})
    .filter(pl.col("gh_project_name") == PROJECT_NAME)
    # --- CORREZIONE: Utilizzo di tr_log_setup_time ---
    .select(["tr_build_id", "tr_job_id", "git_trigger_commit", "gh_build_started_at", "tr_status", "tr_log_setup_time", "tr_log_buildduration"])
    .with_columns([
        pl.col("tr_log_setup_time").cast(pl.Float64, strict=False),
        pl.col("tr_log_buildduration").cast(pl.Float64, strict=False),
        (pl.col("gh_build_started_at").str.to_datetime(strict=False).dt.timestamp("ms") / 1000.0).alias("start_ts")
    ])
    .drop_nulls(subset=["tr_log_setup_time", "tr_log_buildduration", "start_ts"])
).collect()

# --- Dati Test (Netto Setup) ---
df_passed = (
    df.filter(pl.col("tr_status") == "passed")
    # --- CORREZIONE: Calcolo usando tr_log_setup_time ---
    .with_columns([(pl.col("tr_log_buildduration") - pl.col("tr_log_setup_time")).alias("test_duration_net")])
)

test_times = df_passed.filter(pl.col("test_duration_net") > 0)["test_duration_net"].to_numpy()
shape_t, loc_t, scale_t = stats.lognorm.fit(test_times, floc=0)
D_test, _ = stats.kstest(test_times, 'lognorm', args=(shape_t, loc_t, scale_t))

# --- Dati Inter-arrivi (Esogeni) ---
df_builds = (
    df.group_by(["tr_build_id", "git_trigger_commit"])
    .agg([
        pl.col("start_ts").min().alias("build_start"),
        ((pl.col("tr_status") == "failed") | (pl.col("tr_status") == "errored")).sum().alias("bad_jobs_count")
    ])
    .with_columns([(pl.col("bad_jobs_count") > 0).alias("is_bad_build")])
    .sort(["git_trigger_commit", "build_start"])
)

df_arrivals = (
    df_builds.with_columns([pl.col("is_bad_build").shift(1).over("git_trigger_commit").alias("prev_build_bad")])
    .with_columns([(pl.col("prev_build_bad").is_null() | (pl.col("prev_build_bad") == False)).alias("is_new_arrival")])
    .filter(pl.col("is_new_arrival") == True)
    .sort("build_start")
)

inter_arrivals = np.diff(df_arrivals["build_start"].to_numpy())
inter_arrivals = inter_arrivals[inter_arrivals > 0]
loc_a, scale_a = stats.expon.fit(inter_arrivals, floc=0)
D_arr, _ = stats.kstest(inter_arrivals, 'expon', args=(loc_a, scale_a))

# ==========================================
# 2. CREAZIONE DASHBOARD GRAFICA
# ==========================================
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# Limiti per la visualizzazione pulita
test_plot = test_times[test_times < np.percentile(test_times, 98)]
arr_plot = inter_arrivals[inter_arrivals < np.percentile(inter_arrivals, 95)]

# --- PLOT 1: Grafico Cartesiano - Tempi di Test (Sequenza) ---
axes[0, 0].scatter(range(len(test_times)), test_times, color='#3498db', alpha=0.3, s=8, edgecolor='none')
axes[0, 0].set_title("Sequenza Tempi di Test (Netto Setup)", fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel("Indice Job Passed", fontsize=11)
axes[0, 0].set_ylabel("Durata Test (Secondi)", fontsize=11)

# --- PLOT 2: Grafico Cartesiano - Inter-arrivi (Sequenza) ---
axes[0, 1].scatter(range(len(inter_arrivals)), inter_arrivals, color='#e74c3c', alpha=0.3, s=8, edgecolor='none')
axes[0, 1].set_title("Sequenza Inter-arrivi (Solo Traffico Esogeno)", fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel("Indice Build", fontsize=11)
axes[0, 1].set_ylabel("Inter-arrivo (Secondi)", fontsize=11)
axes[0, 1].set_yscale('log')

# --- PLOT 3: Grafico a Barre (Istogramma) - Fitting Lognormale ---
axes[1, 0].hist(test_plot, bins=50, density=True, alpha=0.6, color='#2980b9', edgecolor='white', label='Dati (Top 98%)')
x_test = np.linspace(0, max(test_plot), 200)
axes[1, 0].plot(x_test, stats.lognorm.pdf(x_test, shape_t, loc_t, scale_t), 'k--', lw=2.5, label=f'Fit Lognormale\n(D={D_test:.4f})')
axes[1, 0].set_title("Distribuzione Tempi di Test vs Curva Lognormale", fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel("Durata Test (Secondi)", fontsize=11)
axes[1, 0].set_ylabel("Densità", fontsize=11)
axes[1, 0].legend()

# --- PLOT 4: Grafico a Barre (Istogramma) - Fitting Esponenziale ---
axes[1, 1].hist(arr_plot, bins=50, density=True, alpha=0.6, color='#c0392b', edgecolor='white', label='Dati (Top 95%)')
x_arr = np.linspace(0, max(arr_plot), 200)
axes[1, 1].plot(x_arr, stats.expon.pdf(x_arr, loc_a, scale_a), 'k--', lw=2.5, label=f'Fit Esponenziale\n(D={D_arr:.4f})')
axes[1, 1].set_title("Distribuzione Inter-arrivi vs Curva Esponenziale", fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel("Inter-arrivo (Secondi)", fontsize=11)
axes[1, 1].set_ylabel("Densità", fontsize=11)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print("--- PARAMETRI LOGNORMALE (Fase 2: Tempi di Test Netti) ---")
print(f"Shape (sigma):  {shape_t:.4f}")
print(f"Loc (shift):    {loc_t:.4f}")
print(f"Scale:          {scale_t:.4f}")
print(f"Mu (ln(scale)): {np.log(scale_t):.4f}")

print("\n--- PARAMETRI ESPONENZIALE (Traffico Esogeno Inter-arrivi) ---")
print(f"Loc (shift):          {loc_a:.4f}")
print(f"Scale (Media, 1/λ): {scale_a:.4f}")
print(f"Tasso λ (Arrivi/sec): {1/scale_a:.6f}")

--- PARAMETRI LOGNORMALE (Fase 2: Tempi di Test Netti) ---
Shape (sigma):  0.5295
Loc (shift):    0.0000
Scale:          383.9175
Mu (ln(scale)): 5.9504

--- PARAMETRI ESPONENZIALE (Traffico Esogeno Inter-arrivi) ---
Loc (shift):          0.0000
Scale (Media, 1/λ): 22042.3060
Tasso λ (Arrivi/sec): 0.000045


In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

file_path = '/content/drive/MyDrive/Colab Notebooks/PMCSN/TravisTorrent.csv'
PROJECT_NAME = "SonarSource/sonar-java"

print(f"Estrazione tempi di setup per {PROJECT_NAME}...")

# ==========================================
# 1. ESTRAZIONE DATI SETUP
# ==========================================
df_setup = (
    pl.scan_csv(file_path, null_values=["NA", ""], infer_schema_length=10000,
        schema_overrides={"tr_log_buildduration": pl.String, "tr_log_setup_time": pl.String})
    .filter(pl.col("gh_project_name") == PROJECT_NAME)
    .select(["tr_log_setup_time"])
    .with_columns(pl.col("tr_log_setup_time").cast(pl.Float64, strict=False))
    .drop_nulls()
    # Consideriamo solo i tempi di setup validi (maggiori di zero)
    .filter(pl.col("tr_log_setup_time") > 0)
).collect()

setup_times = df_setup["tr_log_setup_time"].to_numpy()

print(f"Trovati {len(setup_times)} tempi di setup validi.")

# ==========================================
# 2. CREAZIONE BOX PLOT VERTICALE
# ==========================================
sns.set_theme(style="whitegrid")
plt.figure(figsize=(6, 8))

# Filtro visivo per non schiacciare il box plot a causa degli outlier infrastrutturali
percentile_limite = 98
limite_visivo = np.percentile(setup_times, percentile_limite)
setup_times_plot = setup_times[setup_times < limite_visivo]

# Creazione del Box Plot
sns.boxplot(
    y=setup_times_plot,
    color="#f1c40f",       # Un giallo/oro per distinguere il setup dal test
    fliersize=4,           # Dimensione dei punti outlier
    linewidth=1.5,
    width=0.4              # Larghezza della scatola
)

plt.title(f"Box Plot dei Tempi di Setup\n(Filtro visivo al {percentile_limite}° percentile)", fontsize=13, fontweight='bold')
plt.ylabel("Tempo di Setup (Secondi)", fontsize=11)

# Aggiungiamo qualche statistica testuale sul grafico per comodità
mediana = np.median(setup_times)
media = np.mean(setup_times)
plt.text(
    0.25, mediana,
    f"Mediana: {mediana:.1f}s\nMedia: {media:.1f}s",
    verticalalignment='center',
    bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray', boxstyle='round,pad=0.5')
)

plt.tight_layout()
plt.show()

In [ ]:
import polars as pl

file_path = '/content/drive/MyDrive/Colab Notebooks/PMCSN/TravisTorrent.csv'
PROJECT_NAME = "SonarSource/sonar-java"

print(f"--- FREQUENZA DEI TEMPI DI SETUP PER {PROJECT_NAME} ---\n")

# 1. Estrazione dati (come prima)
df_setup = (
    pl.scan_csv(file_path, null_values=["NA", ""], infer_schema_length=10000,
        schema_overrides={"tr_log_buildduration": pl.String, "tr_log_setup_time": pl.String})
    .filter(pl.col("gh_project_name") == PROJECT_NAME)
    .select(["tr_log_setup_time"])
    .with_columns(pl.col("tr_log_setup_time").cast(pl.Float64, strict=False))
    .drop_nulls()
    .filter(pl.col("tr_log_setup_time") > 0)
).collect()

total_setups = df_setup.height

# 2. Raggruppamento e calcolo percentuali
df_freq = (
    df_setup
    .group_by("tr_log_setup_time")
    .agg(pl.len().alias("conteggio"))
    .with_columns([
        ((pl.col("conteggio") / total_setups) * 100).round(4).alias("percentuale_%")
    ])
    # Ordiniamo per frequenza per vedere subito i tempi più ricorrenti
    .sort("conteggio", descending=True)
)

print(f"Totale record di setup validi: {total_setups}")
print(f"Valori unici di setup trovati: {df_freq.height}\n")

print("Tabella delle occorrenze (Top 30 valori più frequenti):")
# Aumentiamo il limite di visualizzazione di Polars per questa cella
pl.Config.set_tbl_rows(30)
print(df_freq.head(30))

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import scipy.stats as stats

PROJECT_NAME = "SonarSource/sonar-java"
file_path = '/content/drive/MyDrive/Colab Notebooks/PMCSN/TravisTorrent.csv'

print("--- ESTRAZIONE TEMPI SEPARATI: SETUP vs TEST (NETTO) ---")

# 1. Estrazione dati con la colonna tr_setup_time
df_proj = (
    pl.scan_csv(file_path, null_values=["NA", ""], infer_schema_length=10000,
        schema_overrides={"tr_log_buildduration": pl.String, "tr_log_setup_time": pl.String})
    .filter(pl.col("gh_project_name") == PROJECT_NAME)
    .select(["tr_build_id", "tr_status", "tr_log_setup_time", "tr_log_buildduration"])
    .with_columns([
        pl.col("tr_log_setup_time").cast(pl.Float64, strict=False),
        pl.col("tr_log_buildduration").cast(pl.Float64, strict=False)
    ])
    .drop_nulls()
    # Calcoliamo la durata netta della compilazione/test
    .with_columns([
        (pl.col("tr_log_buildduration") - pl.col("tr_log_setup_time")).alias("test_duration")
    ])
    # Pulizia: teniamo solo i record in cui entrambe le fasi hanno richiesto tempo
    .filter((pl.col("tr_log_setup_time") > 0) & (pl.col("test_duration") > 0))
    .collect().to_pandas()
)

# 2. Distribuzione del SETUP (Calcolata su TUTTI i job)
setup_times = df_proj["tr_log_setup_time"].values
shape_s, loc_s, scale_s = stats.lognorm.fit(setup_times, floc=0)
D_setup, p_setup = stats.kstest(setup_times, 'lognorm', args=(shape_s, loc_s, scale_s))

# 3. Distribuzione del TEST (Calcolata SOLO sui job 'PASSED')
df_passed = df_proj[df_proj["tr_status"] == "passed"]
test_times = df_passed["test_duration"].values
shape_t, loc_t, scale_t = stats.lognorm.fit(test_times, floc=0)
D_test, p_test = stats.kstest(test_times, 'lognorm', args=(shape_t, loc_t, scale_t))

print(f"\n[FASE 1: SETUP] (Campioni: {len(setup_times)})")
print(f"  Lognormale -> Mu (ln scale): {np.log(scale_s):.4f} | Sigma (shape): {shape_s:.4f}")
print(f"  Media Empirica: {np.mean(setup_times):.2f} secondi")
print(f"  Errore KS (D): {D_setup:.4f}")

print(f"\n[FASE 2: TEST IDEALE NETTO] (Campioni: {len(test_times)})")
print(f"  Lognormale -> Mu (ln scale): {np.log(scale_t):.4f} | Sigma (shape): {shape_t:.4f}")
print(f"  Media Empirica: {np.mean(test_times):.2f} secondi")
print(f"  Errore KS (D): {D_test:.4f}")

In [ ]:
import polars as pl

file_path = '/content/drive/MyDrive/Colab Notebooks/PMCSN/TravisTorrent.csv'
PROJECT_NAME = "SonarSource/sonar-java"

print(f"Estrazione dei record per commit trigger sul progetto: {PROJECT_NAME}...\n")

# Creazione della query con LazyFrame per ottimizzare la RAM
df_commits = (
    pl.scan_csv(
        file_path,
        null_values=["NA", ""],
        infer_schema_length=10000
    )
    .filter(pl.col("gh_project_name") == PROJECT_NAME)

    # Raggruppiamo per l'hash del commit
    .group_by("git_trigger_commit")
    .agg([
        # Conta il numero totale di record (job) scatenati da questo commit
        pl.len().alias("totale_job"),

        # Conta quante build distinte sono state generate
        pl.col("tr_build_id").n_unique().alias("build_uniche")
    ])

    # Ordiniamo in modo decrescente per vedere i commit più "pesanti"
    .sort("totale_job", descending=True)
).collect()

# Stampa dei risultati
with pl.Config(tbl_rows=20):
    print("--- TOP 20 COMMIT PER NUMERO DI JOB GENERATI ---")
    print(df_commits.head(20))

# Stampa del riepilogo generale
print(f"\nNumero totale di commit unici (trigger) trovati: {df_commits.height}")

Estrazione dei record per commit trigger sul progetto: SonarSource/sonar-java...

--- TOP 20 COMMIT PER NUMERO DI JOB GENERATI ---
shape: (20, 3)
┌─────────────────────────────────┬────────────┬──────────────┐
│ git_trigger_commit              ┆ totale_job ┆ build_uniche │
│ ---                             ┆ ---        ┆ ---          │
│ str                             ┆ u32        ┆ u32          │
╞═════════════════════════════════╪════════════╪══════════════╡
│ ec579aff0ea63c2ab37006e926f4e8… ┆ 126        ┆ 14           │
│ 445b488bcdd30dc45b17420ceb888a… ┆ 90         ┆ 10           │
│ 4d6a2f3d81f946032941d88b17a92a… ┆ 81         ┆ 9            │
│ d0a7c1f35a59b6956222fd08f12d86… ┆ 72         ┆ 9            │
│ 717c052673ad6e79b35522d423b9f9… ┆ 72         ┆ 9            │
│ 830d17cb913b543d4d20ff23edb4ee… ┆ 70         ┆ 7            │
│ 90411d927d27a33caa8f1b0ab22831… ┆ 64         ┆ 8            │
│ 0de67703b4dfd8b857e4eb89e5d361… ┆ 64         ┆ 8            │
│ 56513b8ce2eb1e9a21c0

In [ ]:
import polars as pl

file_path = '/content/drive/MyDrive/Colab Notebooks/PMCSN/TravisTorrent.csv'
PROJECT_NAME = "SonarSource/sonar-java"

print(f"Analisi dei commit e degli stati dei job per il progetto: {PROJECT_NAME}...\n")

df_commits = (
    pl.scan_csv(
        file_path,
        null_values=["NA", ""],
        infer_schema_length=10000
    )
    .filter(pl.col("gh_project_name") == PROJECT_NAME)

    # Raggruppiamo sempre per l'hash del commit
    .group_by("git_trigger_commit")
    .agg([
        # Metriche generali
        pl.len().alias("totale_job"),
        pl.col("tr_build_id").n_unique().alias("build_uniche"),

        # Conteggio specifico per ogni stato
        (pl.col("tr_status") == "passed").sum().alias("job_passed"),
        (pl.col("tr_status") == "failed").sum().alias("job_failed"),
        (pl.col("tr_status") == "canceled").sum().alias("job_canceled"),
        (pl.col("tr_status") == "errored").sum().alias("job_errored")
    ])

    # Ordiniamo in modo decrescente per vedere i commit che hanno lavorato di più
    .sort("totale_job", descending=True)
).collect()

# Stampa dei risultati
with pl.Config(tbl_rows=20):
    print("--- TOP 20 COMMIT PER NUMERO DI JOB CON DETTAGLIO STATI ---")
    print(df_commits.head(20))

print(f"\nNumero totale di commit unici (trigger) trovati: {df_commits.height}")

Analisi dei commit e degli stati dei job per il progetto: SonarSource/sonar-java...

--- TOP 20 COMMIT PER NUMERO DI JOB CON DETTAGLIO STATI ---
shape: (20, 7)
┌───────────────┬────────────┬──────────────┬────────────┬────────────┬──────────────┬─────────────┐
│ git_trigger_c ┆ totale_job ┆ build_uniche ┆ job_passed ┆ job_failed ┆ job_canceled ┆ job_errored │
│ ommit         ┆ ---        ┆ ---          ┆ ---        ┆ ---        ┆ ---          ┆ ---         │
│ ---           ┆ u32        ┆ u32          ┆ u32        ┆ u32        ┆ u32          ┆ u32         │
│ str           ┆            ┆              ┆            ┆            ┆              ┆             │
╞═══════════════╪════════════╪══════════════╪════════════╪════════════╪══════════════╪═════════════╡
│ ec579aff0ea63 ┆ 126        ┆ 14           ┆ 126        ┆ 0          ┆ 0            ┆ 0           │
│ c2ab37006e926 ┆            ┆              ┆            ┆            ┆              ┆             │
│ f4e8…         ┆            ┆  

In [ ]:
import json

# ... (codice precedente per calcolare df_commits) ...

# 1. Convertiamo il DataFrame Polars in una stringa JSON riga per riga
json_string = df_commits.write_json()

# 2. Formattiamo la stringa con un'indentazione di 4 spazi per una facile lettura
json_formattato = json.dumps(json.loads(json_string), indent=4)

print("\n--- RISULTATO IN FORMATO JSON ---")
print(json_formattato)


--- RISULTATO IN FORMATO JSON ---
[
    {
        "git_trigger_commit": "ec579aff0ea63c2ab37006e926f4e83e5cd13ea4",
        "totale_job": 126,
        "build_uniche": 14,
        "job_passed": 126,
        "job_failed": 0,
        "job_canceled": 0,
        "job_errored": 0
    },
    {
        "git_trigger_commit": "445b488bcdd30dc45b17420ceb888a15032ad5ef",
        "totale_job": 90,
        "build_uniche": 10,
        "job_passed": 90,
        "job_failed": 0,
        "job_canceled": 0,
        "job_errored": 0
    },
    {
        "git_trigger_commit": "4d6a2f3d81f946032941d88b17a92a0fe04a9218",
        "totale_job": 81,
        "build_uniche": 9,
        "job_passed": 54,
        "job_failed": 0,
        "job_canceled": 0,
        "job_errored": 27
    },
    {
        "git_trigger_commit": "717c052673ad6e79b35522d423b9f979441492dc",
        "totale_job": 72,
        "build_uniche": 9,
        "job_passed": 72,
        "job_failed": 0,
        "job_canceled": 0,
        "job_error

In [ ]:
import polars as pl
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

file_path = '/content/drive/MyDrive/Colab Notebooks/PMCSN/TravisTorrent.csv'
PROJECT_NAME = "SonarSource/sonar-java"

print("Estrazione dei tempi di feedback umano (Restart Delay)...")

# 1. Estrazione dati di base.
#    Fix 6: schema_overrides + cast strict=False per robustezza sul parsing di buildduration.
#    Aggiunto tr_status per identificare le build "cattive" (>=1 job failed/errored).
df = (
    pl.scan_csv(file_path, null_values=["NA", ""], infer_schema_length=10000,
                schema_overrides={"tr_log_buildduration": pl.String})
    .filter(pl.col("gh_project_name") == PROJECT_NAME)
    .select(["git_trigger_commit", "tr_build_id", "gh_build_started_at", "tr_log_buildduration", "tr_status"])
    .drop_nulls(subset=["git_trigger_commit", "tr_build_id", "gh_build_started_at"])
    .with_columns([
        pl.col("tr_log_buildduration").cast(pl.Float64, strict=False),
        (pl.col("gh_build_started_at").str.to_datetime(strict=False).dt.timestamp("ms") / 1000.0).alias("start_ts")
    ])
).collect()

# 2. Raggruppamento per singola build, con flag di build "cattiva".
df_builds = (
    df.group_by(["git_trigger_commit", "tr_build_id"])
    .agg([
        pl.col("start_ts").min().alias("build_start"),
        pl.col("tr_log_buildduration").max().alias("max_duration"),  # la build finisce col job piu' lungo
        ((pl.col("tr_status") == "failed") | (pl.col("tr_status") == "errored")).sum().alias("bad_jobs_count")
    ])
    .with_columns([
        # durata mancante (build interamente errored) -> fine ~ inizio
        (pl.col("build_start") + pl.col("max_duration").fill_null(0.0)).alias("build_end"),
        (pl.col("bad_jobs_count") > 0).alias("is_bad_build")
    ])
    .sort(["git_trigger_commit", "build_start"])
)

# 3. Gap tra questa build e la successiva (stesso commit).
#    Fix 3: il feedback umano avviene solo DOPO un fallimento -> teniamo solo i gap
#    in cui la build CORRENTE (da cui parte il gap) e' "cattiva".
df_feedback = (
    df_builds
    .with_columns(
        pl.col("build_start").shift(-1).over("git_trigger_commit").alias("next_build_start")
    )
    .with_columns(
        (pl.col("next_build_start") - pl.col("build_end")).alias("feedback_time")
    )
    .filter((pl.col("feedback_time") > 0) & (pl.col("is_bad_build") == True))
)

feedback_times = df_feedback["feedback_time"].to_numpy()
print(f"Trovati {len(feedback_times)} eventi di 'Restart' umano validi (solo dopo fallimenti).")

# Rimuoviamo gli outlier estremi (restart entro 48 ore = 172800 secondi)
feedback_times = feedback_times[feedback_times < 172800]

# Fix 1: il simulatore interpreta la lognormale in MINUTI (delay_minutes * 60),
# quindi il fit va fatto sui tempi espressi in MINUTI.
feedback_min = feedback_times / 60.0

# 4. Fitting Statistico (Lognormale sui tempi in minuti)
shape_h, loc_h, scale_h = stats.lognorm.fit(feedback_min, floc=0)

# Stampa Parametri per config.json (human_feedback)
print("\n--- PARAMETRI DEL NODO DI RITARDO UMANO (in MINUTI) ---")
print(f"Distribuzione:         Lognormale")
print(f"sigma_delay (shape):   {shape_h:.4f}")
print(f"scale (mediana, min):  {scale_h:.4f}")
print(f"mu_delay = ln(scale):  {np.log(scale_h):.4f}")
print(f"Media teorica:         {scale_h * np.exp((shape_h**2)/2):.2f} minuti")

# 5. Visualizzazione (in minuti)
plt.figure(figsize=(10, 5))
plt.hist(feedback_min, bins=60, density=True, alpha=0.6, color='#27ae60', label='Dati Reali (Feedback Time)')
x = np.linspace(0, max(feedback_min), 300)
plt.plot(x, stats.lognorm.pdf(x, shape_h, loc_h, scale_h), 'k--', lw=2, label='Fit Lognormale')
plt.title("Tempo di Reazione Umano ai Fallimenti (Think Time)", fontsize=13, fontweight='bold')
plt.xlabel("Ritardo (Minuti)", fontsize=11)
plt.ylabel("Densita'", fontsize=11)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import polars as pl
import numpy as np
import scipy.stats as stats

file_path = '/content/drive/MyDrive/Colab Notebooks/PMCSN/TravisTorrent.csv'
PROJECT_NAME = "SonarSource/sonar-java"

print("--- 1. CALCOLO PROBABILITA' STATI DEI JOB ---")

# Fix 5: le probabilita' di routing si calcolano su TUTTI i job del progetto, senza
# drop_nulls su colonne non pertinenti (commit/timestamp) che altererebbero i denominatori.
df_routing = (
    pl.scan_csv(file_path, null_values=["NA", ""], infer_schema_length=10000)
    .filter(pl.col("gh_project_name") == PROJECT_NAME)
    .select(["tr_status"])
    .drop_nulls()
).collect()

total_jobs = df_routing.height
df_probs = (
    df_routing.group_by("tr_status")
    .agg(pl.len().alias("count"))
    .with_columns(((pl.col("count") / total_jobs)).alias("probabilita"))
    .sort("probabilita", descending=True)
)
print(f"Totale job analizzati: {total_jobs}")
print(df_probs)

print("\n--- 2. RICALCOLO TEMPI DI INTER-ARRIVO (SOLO NUOVI ARRIVI) ---")

# Per gli inter-arrivi servono commit e timestamp: qui il drop_nulls e' legittimo.
df_jobs = (
    pl.scan_csv(file_path, null_values=["NA", ""], infer_schema_length=10000)
    .filter(pl.col("gh_project_name") == PROJECT_NAME)
    .select(["tr_build_id", "tr_job_id", "git_trigger_commit", "gh_build_started_at", "tr_status"])
    .drop_nulls()
    .with_columns([
        (pl.col("gh_build_started_at").str.to_datetime(strict=False).dt.timestamp("ms") / 1000.0).alias("start_ts")
    ])
).collect()

# Raggruppiamo i job in build per capire lo stato globale della matrice
df_builds = (
    df_jobs.group_by(["tr_build_id", "git_trigger_commit"])
    .agg([
        pl.col("start_ts").min().alias("build_start"),
        ((pl.col("tr_status") == "failed") | (pl.col("tr_status") == "errored")).sum().alias("bad_jobs_count")
    ])
    .with_columns([
        (pl.col("bad_jobs_count") > 0).alias("is_bad_build")
    ])
    .sort(["git_trigger_commit", "build_start"])
)

# Identifichiamo i veri "Nuovi Arrivi": prima build del commit, o build dopo una PASSATA.
df_builds_filtered = (
    df_builds
    .with_columns([
        pl.col("is_bad_build").shift(1).over("git_trigger_commit").alias("prev_build_bad")
    ])
    .with_columns([
        (pl.col("prev_build_bad").is_null() | (pl.col("prev_build_bad") == False)).alias("is_new_arrival")
    ])
)

new_arrivals_ts = (
    df_builds_filtered
    .filter(pl.col("is_new_arrival") == True)
    .sort("build_start")
)["build_start"].to_numpy()

inter_arrivi = np.diff(new_arrivals_ts)
inter_arrivi = inter_arrivi[inter_arrivi > 0]

loc_a, scale_a = stats.expon.fit(inter_arrivi, floc=0)
D_arr, p_arr = stats.kstest(inter_arrivi, 'expon', args=(loc_a, scale_a))

print(f"Build totali: {df_builds.height}")
print(f"Nuovi arrivi effettivi (filtrati i riavvii manuali): {len(new_arrivals_ts)}")
print(f"\n--- FIT ESPONENZIALE (Traffico Esogeno) ---")
print(f"Scale (1/lambda, Media inter-arrivo): {scale_a:.2f} secondi")
print(f"Tasso lambda (Arrivi al secondo):     {1/scale_a:.6f}")
print(f"KS-Test Errore (D):                   {D_arr:.4f}")

In [ ]:
import polars as pl

file_path = '/content/drive/MyDrive/Colab Notebooks/PMCSN/TravisTorrent.csv'
PROJECT_NAME = "SonarSource/sonar-java"

print(f"--- CALCOLO PROBABILITA' DI RETRY (VERO) PER {PROJECT_NAME} ---\n")

# 1. Caricamento. Fix 2: NON richiediamo tr_log_buildduration nel drop_nulls, altrimenti
#    si scarterebbe l'87% dei job errored (che non hanno buildduration). Fix 6: schema robusto.
df = (
    pl.scan_csv(file_path, null_values=["NA", ""], infer_schema_length=10000,
                schema_overrides={"tr_log_buildduration": pl.String})
    .filter(pl.col("gh_project_name") == PROJECT_NAME)
    .select(["tr_build_id", "tr_job_id", "git_trigger_commit", "gh_build_started_at", "tr_status", "tr_log_buildduration"])
    .drop_nulls(subset=["gh_build_started_at", "git_trigger_commit"])
    .with_columns([
        pl.col("tr_log_buildduration").cast(pl.Float64, strict=False),
        (pl.col("gh_build_started_at").str.to_datetime(strict=False).dt.timestamp("ms") / 1000.0).alias("start_ts")
    ])
    .with_columns([
        # end_ts = morte del job; buildduration mancante (errori infra) -> durata 0
        (pl.col("start_ts") + pl.col("tr_log_buildduration").fill_null(0.0)).alias("end_ts")
    ])
).collect()

# 2. Ultimo avvio in assoluto per un dato commit
df_max_start = (
    df.group_by("git_trigger_commit")
    .agg(pl.col("start_ts").max().alias("ultimo_avvio_del_commit"))
)

# 3. Un VERO retry: esiste un avvio successivo sullo stesso commit dopo la morte del job (+10s margine)
df_retry = (
    df.join(df_max_start, on="git_trigger_commit")
    .with_columns([
        (pl.col("ultimo_avvio_del_commit") > (pl.col("end_ts") + 10)).alias("is_retried")
    ])
)

# 4. Probabilita' per stato
stati_anomali = ["failed", "errored"]

for stato in stati_anomali:
    df_stato = df_retry.filter(pl.col("tr_status") == stato)
    totale = df_stato.height
    retried = df_stato.filter(pl.col("is_retried") == True).height

    if totale > 0:
        p_retry = retried / totale
        p_abbandono = 1.0 - p_retry

        print(f"Stato [{stato.upper()}]:")
        print(f"  Totale occorrenze:   {totale}")
        print(f"  Di cui riavviate:    {retried}")
        print(f"  -> Probabilita' di Retry VERO (Feedback): {p_retry:.4f} ({p_retry*100:.2f}%)")
        print(f"  -> Probabilita' di Abbandono (Sink): {p_abbandono:.4f} ({p_abbandono*100:.2f}%)\n")

In [ ]:
import polars as pl

file_path = '/content/drive/MyDrive/Colab Notebooks/PMCSN/TravisTorrent.csv'
PROJECT_NAME = "SonarSource/sonar-java"

print(f"--- ESTRAZIONE PMF: DIMENSIONE DEL BATCH (JOB PER BUILD) PER {PROJECT_NAME} ---\n")

# 1. Caricamento dati e conteggio dei job per ogni singola build
df_builds = (
    pl.scan_csv(file_path, null_values=["NA", ""], infer_schema_length=10000)
    .filter(pl.col("gh_project_name") == PROJECT_NAME)
    .select(["tr_build_id", "tr_job_id"])
    .drop_nulls()
    # Raggruppiamo per build e contiamo quanti job contiene
    .group_by("tr_build_id")
    .agg(pl.len().alias("jobs_per_build"))
).collect()

total_builds = df_builds.height

# 2. Calcolo della Probability Mass Function (PMF)
df_pmf = (
    df_builds
    .group_by("jobs_per_build")
    .agg(pl.len().alias("frequenza_assoluta"))
    .with_columns([
        # Calcolo probabilità (0-1) utile per il simulatore
        (pl.col("frequenza_assoluta") / total_builds).round(4).alias("probabilità"),
        # Calcolo percentuale per la stesura del report
        ((pl.col("frequenza_assoluta") / total_builds) * 100).round(2).alias("percentuale_%")
    ])
    # Ordiniamo dal numero di job più piccolo al più grande
    .sort("jobs_per_build")
)

print(f"Totale build analizzate: {total_builds}\n")
print("Probability Mass Function (PMF) della dimensione del batch:")

# Impostiamo il limite di visualizzazione di Polars per vedere tutta la tabella
pl.Config.set_tbl_rows(30)
print(df_pmf)

--- ESTRAZIONE PMF: DIMENSIONE DEL BATCH (JOB PER BUILD) PER SonarSource/sonar-java ---

Totale build analizzate: 2278

Probability Mass Function (PMF) della dimensione del batch:
shape: (9, 4)
┌────────────────┬────────────────────┬─────────────┬───────────────┐
│ jobs_per_build ┆ frequenza_assoluta ┆ probabilità ┆ percentuale_% │
│ ---            ┆ ---                ┆ ---         ┆ ---           │
│ u32            ┆ u32                ┆ f64         ┆ f64           │
╞════════════════╪════════════════════╪═════════════╪═══════════════╡
│ 1              ┆ 97                 ┆ 0.0426      ┆ 4.26          │
│ 2              ┆ 43                 ┆ 0.0189      ┆ 1.89          │
│ 3              ┆ 104                ┆ 0.0457      ┆ 4.57          │
│ 4              ┆ 172                ┆ 0.0755      ┆ 7.55          │
│ 6              ┆ 241                ┆ 0.1058      ┆ 10.58         │
│ 7              ┆ 178                ┆ 0.0781      ┆ 7.81          │
│ 8              ┆ 790              